In [47]:
## Demographic Clustering

In [20]:
import pandas as pd
import numpy as np
import sys
!{sys.executable} -m pip install scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
df = pd.read_csv(r"C:\Users\liang\Desktop\Work\Project\GAD7\GAD7_clean.csv")

In [19]:
## 1. DATASET PREPROCESS

df_cluster = df[['age', 'marriage', 'income', 'education', 'gender']].copy()

# age
df_cluster['age'] = pd.to_numeric(df_cluster['age'], errors='coerce')
df_cluster['age'] = df_cluster['age'].fillna(df_cluster['age'].median())

# marriage
df_cluster['marriage'] = pd.to_numeric(df_cluster['marriage'], errors='coerce')
df_cluster['marriage'] = df_cluster['marriage'].replace({4: 3})

# Convert str
for col in ['income', 'education', 'gender']:
    df_cluster[col] = pd.to_numeric(df_cluster[col], errors='coerce')

# check null
print("Missing values:")
print(df_cluster.isna().sum())

print("\nData shape:", df_cluster.shape)
print("\nPreview:")
print(df_cluster.head())

print("\nValue counts for marriage:")
print(df_cluster['marriage'].value_counts().sort_index())

Missing values:
age          0
marriage     0
income       0
education    0
gender       0
dtype: int64

Data shape: (30000, 5)

Preview:
    age  marriage  income  education  gender
0  26.0         0       3          4       1
1  36.0         2       2          5       1
2  11.0         0       0          2       0
3  10.0         0       0          1       1
4  10.0         0       0          1       1

Value counts for marriage:
marriage
0    15401
1     1468
2    11848
3     1283
Name: count, dtype: int64


In [18]:
## 2. ONE-HOT ENCODING AND STANDARDIZATION


# 1. Split numerical and catagorical data

numeric_col = ['age']
categorical_cols = ['marriage', 'income', 'education', 'gender']


# 2. One-hot encoding
df_encoded = pd.get_dummies(
    df_cluster,
    columns=categorical_cols,
    drop_first=False
)


# 3. Standardize age
scaler = StandardScaler()
df_encoded[numeric_col] = scaler.fit_transform(df_encoded[numeric_col])


# 4. Check result
print("Encoded data shape:", df_encoded.shape)
print("\nColumns:")
print(df_encoded.columns.tolist())

print("\nPreview:")
print(df_encoded.head())

## 5. Convert boolean to 0/1
X = df_encoded.copy()

bool_cols = X.select_dtypes(include='bool').columns
X[bool_cols] = X[bool_cols].astype(int)

print(X.dtypes)
print(X.head())

Encoded data shape: (30000, 20)

Columns:
['age', 'marriage_0', 'marriage_1', 'marriage_2', 'marriage_3', 'income_0', 'income_1', 'income_2', 'income_3', 'income_4', 'income_5', 'education_0', 'education_1', 'education_2', 'education_3', 'education_4', 'education_5', 'education_6', 'gender_0', 'gender_1']

Preview:
        age  marriage_0  marriage_1  marriage_2  marriage_3  income_0  \
0  0.011510        True       False       False       False     False   
1  1.261652       False       False        True       False     False   
2 -1.863703        True       False       False       False      True   
3 -1.988718        True       False       False       False      True   
4 -1.988718        True       False       False       False      True   

   income_1  income_2  income_3  income_4  income_5  education_0  education_1  \
0     False     False      True     False     False        False        False   
1     False      True     False     False     False        False        False   
2

In [26]:
## 3. K Scan
k_values = range(2, 9)
silhouette_scores = []
inertias = []
cluster_sizes = {}

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)

    sil = silhouette_score(X, labels)
    inertia = kmeans.inertia_

    silhouette_scores.append(sil)
    inertias.append(inertia)
    cluster_sizes[k] = pd.Series(labels).value_counts().sort_index()

    print(f"\nK = {k}")
    print(f"Silhouette Score = {sil:.4f}")
    print(f"Inertia = {inertia:.2f}")
    print("Cluster sizes:")
    print(cluster_sizes[k])


K = 2
Silhouette Score = 0.2612
Inertia = 73328.88
Cluster sizes:
0    12438
1    17562
Name: count, dtype: int64

K = 3
Silhouette Score = 0.2032
Inertia = 64982.01
Cluster sizes:
0    12952
1     8171
2     8877
Name: count, dtype: int64

K = 4
Silhouette Score = 0.2019
Inertia = 58437.82
Cluster sizes:
0     5830
1     7748
2    11102
3     5320
Name: count, dtype: int64

K = 5
Silhouette Score = 0.2068
Inertia = 54158.28
Cluster sizes:
0    6502
1    8265
2    5485
3    5702
4    4046
Name: count, dtype: int64

K = 6
Silhouette Score = 0.2178
Inertia = 49765.32
Cluster sizes:
0    3882
1    5745
2    8312
3    3405
4    4454
5    4202
Name: count, dtype: int64

K = 7
Silhouette Score = 0.2219
Inertia = 47184.22
Cluster sizes:
0    3462
1    8064
2    3805
3    3088
4    4454
5    3726
6    3401
Name: count, dtype: int64

K = 8
Silhouette Score = 0.2094
Inertia = 44755.30
Cluster sizes:
0    3401
1    3722
2    2725
3    3723
4    4454
5    2994
6    3465
7    5516
Name: count, dty

In [27]:
## 3. K scan result:
## Cluster solutions from K = 2 to K = 8 were evaluated using silhouette scores, inertia, cluster size balance, and interpretability. 
## Although the highest silhouette score was observed at K = 2, this solution was considered too coarse for meaningful subgroup profiling. 
## A 6-cluster solution was selected as the final model because it provided a reasonable balance between clustering quality and interpretability, 
## while maintaining adequate sample sizes across clusters.

In [33]:
## 4. K = 6 Cluster Profile

# final model
k_final = 6
kmeans_final = KMeans(n_clusters=k_final, random_state=42, n_init=10)
df_cluster['cluster'] = kmeans_final.fit_predict(X)

cluster_counts = df_cluster['cluster'].value_counts().sort_index()
cluster_perc = df_cluster['cluster'].value_counts(normalize=True).sort_index() * 100

print("\nCluster percentages:")
print(cluster_perc.round(2))

# Num Into Str
df_profile = df_cluster.copy()

df_profile['gender_label'] = df_profile['gender'].map({
    0: 'female',
    1: 'male'
})

df_profile['marriage_label'] = df_profile['marriage'].map({
    0: 'single',
    1: 'cohabit',
    2: 'married',
    3: 'divorced/widowed'
})

df_profile['income_label'] = df_profile['income'].map({
    0: '0-50k',
    1: '50-100k',
    2: '100-200k',
    3: '200-400k',
    4: '400-800k',
    5: '800k+'
})

df_profile['education_label'] = df_profile['education'].map({
    0: 'primary',
    1: 'middle school',
    2: 'high school',
    3: 'college',
    4: 'university',
    5: "master's",
    6: 'doctorate'
})

# Age as Numerical
age_summary = df_cluster.groupby('cluster')['age'].agg(['mean', 'median', 'std', 'min', 'max', 'count'])
print(age_summary.round(2))

# Catagorical
def cluster_distribution_table(df, cluster_col, var):
    table = pd.crosstab(df[cluster_col], df[var], normalize='index') * 100
    return table.round(2)

print("Gender distribution by cluster (%)")
print(cluster_distribution_table(df_cluster, 'cluster', 'gender'))

print("\nMarriage distribution by cluster (%)")
print(cluster_distribution_table(df_cluster, 'cluster', 'marriage'))

print("\nIncome distribution by cluster (%)")
print(cluster_distribution_table(df_cluster, 'cluster', 'income'))

print("\nEducation distribution by cluster (%)")
print(cluster_distribution_table(df_cluster, 'cluster', 'education'))


Cluster percentages:
cluster
0    12.94
1    19.15
2    27.71
3    11.35
4    14.85
5    14.01
Name: proportion, dtype: float64
          mean  median   std   min   max  count
cluster                                        
0        23.75    23.0  3.55  10.0  36.0   3882
1        27.46    27.0  3.74  16.0  35.0   5745
2        19.46    19.0  3.37  10.0  33.0   8312
3        29.05    29.0  4.42  16.0  38.0   3405
4        20.99    21.0  3.98  10.0  34.0   4454
5        41.19    41.0  3.75  34.0  48.0   4202
Gender distribution by cluster (%)
gender        0       1
cluster                
0          0.00  100.00
1          0.00  100.00
2          0.00  100.00
3        100.00    0.00
4        100.00    0.00
5         29.11   70.89

Marriage distribution by cluster (%)
marriage      0      1      2      3
cluster                             
0         81.35  12.70   0.00   5.95
1          0.19   1.22  94.20   4.39
2         91.45   6.98   0.12   1.46
3          6.73   4.55  83.38   5.35


In [32]:
## 4. First Cluster Result
## Cluster 0: Young single medium-income men
##            24 years old, Single(81.35%), 50-100k/yr(77.56%), University&College(72%)
## Cluster 1: Young married lower-income men
##            27 years old, Married(94.2%), 0-100k/yr(81%)
## Cluster 2: Very young single low-income men
##            20 years old, Single(91.45%), 0-50k/yr(99.46%)
## Cluster 3: Married middle-income women
##            29 years old, Married(83.4%), 50-100k/yr(43.75%), University&College(54%)
## Cluster 4: Young single low-income women
##            21 years old, Single(96.35%), 0-100k/yr(91%)
## Cluster 5: Older predominantly married adults
##            41 years old, Male(71%), Female(29%), Married(85%)

## Interpretation
## The clustering results suggest that the sample can be divided into several demographic profiles,
## but the solution appears to be strongly driven by gender.
## Within each gender-defined subgroup,age and marital status further separated respondents into younger single groups and older married groups.
## Income differences were also aligned with age and marital structure,
## with the youngest clusters showing the strongest concentration in the lowest income category.
## Therefore, the six-cluster solution is interpretable, but it should be regarded as a gender-dominated demographic segmentation,
## rather than a fully balanced latent population structure.

In [39]:
## 5. Second Cluster With Less One-hot Encoding
df_v1 = df_cluster[['age', 'marriage', 'income', 'education', 'gender']].copy()

numeric_col = ['age']
categorical_cols = ['marriage', 'income', 'education', 'gender']

# one-hot with drop_first=True
df_v1_encoded = pd.get_dummies(
    df_v1,
    columns=categorical_cols,
    drop_first=True
)

# age
scaler_v1 = StandardScaler()
df_v1_encoded[numeric_col] = scaler_v1.fit_transform(df_v1_encoded[numeric_col])

# bool to int
bool_cols = df_v1_encoded.select_dtypes(include='bool').columns
df_v1_encoded[bool_cols] = df_v1_encoded[bool_cols].astype(int)

X_v1 = df_v1_encoded.copy()

X_v1_sample = X_v1.sample(n=5000, random_state=42)

k_values = range(2, 9)
silhouette_scores_v1 = []
inertias_v1 = []
cluster_sizes_v1 = {}

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_full = kmeans.fit_predict(X_v1)
    labels_sample = kmeans.predict(X_v1_sample)

    sil = silhouette_score(X_v1_sample, labels_sample)
    inertia = kmeans.inertia_

    silhouette_scores_v1.append(sil)
    inertias_v1.append(inertia)
    cluster_sizes_v1[k] = pd.Series(labels_full).value_counts().sort_index()

    print(f"\nK = {k}")
    print(f"Silhouette Score = {sil:.4f}")
    print(f"Inertia = {inertia:.2f}")
    print("Cluster sizes:")
    print(cluster_sizes_v1[k])


K = 2
Silhouette Score = 0.2828
Inertia = 56160.32
Cluster sizes:
0     9542
1    20458
Name: count, dtype: int64

K = 3
Silhouette Score = 0.1867
Inertia = 49740.22
Cluster sizes:
0     5217
1    14765
2    10018
Name: count, dtype: int64

K = 4
Silhouette Score = 0.1841
Inertia = 44909.54
Cluster sizes:
0     5084
1    10563
2     8202
3     6151
Name: count, dtype: int64

K = 5
Silhouette Score = 0.2200
Inertia = 40774.04
Cluster sizes:
0    6047
1    7549
2    4934
3    5931
4    5539
Name: count, dtype: int64

K = 6
Silhouette Score = 0.2428
Inertia = 38009.86
Cluster sizes:
0    4029
1    5997
2    6007
3    4030
4    6308
5    3629
Name: count, dtype: int64

K = 7
Silhouette Score = 0.2493
Inertia = 35770.84
Cluster sizes:
0    5381
1    3980
2    3921
3    3653
4    4558
5    3285
6    5222
Name: count, dtype: int64

K = 8
Silhouette Score = 0.2552
Inertia = 34024.21
Cluster sizes:
0    2494
1    4781
2    3157
3    5091
4    3979
5    3884
6    3823
7    2791
Name: count, dty

In [38]:
## 5. Further Information With New Model
# Version 1 final model: drop_first=True
k_final_v1 = 6
kmeans_final_v1 = KMeans(n_clusters=k_final_v1, random_state=42, n_init=10)

df_v1_profile = df_cluster[['age', 'marriage', 'income', 'education', 'gender']].copy()
df_v1_profile['cluster'] = kmeans_final_v1.fit_predict(X_v1)

# cluster counts
cluster_counts_v1 = df_v1_profile['cluster'].value_counts().sort_index()
cluster_perc_v1 = df_v1_profile['cluster'].value_counts(normalize=True).sort_index() * 100

print("\nCluster percentages:")
print(cluster_perc_v1.round(2))

age_summary_v1 = df_v1_profile.groupby('cluster')['age'].agg(['mean', 'median', 'std', 'min', 'max', 'count'])
print(age_summary_v1.round(2))

def cluster_distribution_table(df, cluster_col, var):
    table = pd.crosstab(df[cluster_col], df[var], normalize='index') * 100
    return table.round(2)

print("Gender distribution by cluster (%)")
print(cluster_distribution_table(df_v1_profile, 'cluster', 'gender'))

print("\nMarriage distribution by cluster (%)")
print(cluster_distribution_table(df_v1_profile, 'cluster', 'marriage'))

print("\nIncome distribution by cluster (%)")
print(cluster_distribution_table(df_v1_profile, 'cluster', 'income'))

print("\nEducation distribution by cluster (%)")
print(cluster_distribution_table(df_v1_profile, 'cluster', 'education'))



Cluster percentages:
cluster
0    13.43
1    19.99
2    20.02
3    13.43
4    21.03
5    12.10
Name: proportion, dtype: float64
          mean  median   std   min   max  count
cluster                                        
0        41.55    41.0  3.47  35.0  48.0   4029
1        21.63    21.0  3.17  12.0  32.0   5997
2        22.17    22.0  3.24  10.0  31.0   6007
3        20.05    20.0  4.10  11.0  31.0   4030
4        30.43    30.0  3.02  23.0  36.0   6308
5        20.46    21.0  4.35  10.0  31.0   3629
Gender distribution by cluster (%)
gender       0      1
cluster              
0        36.76  63.24
1        24.26  75.74
2        23.97  76.03
3        31.99  68.01
4        34.07  65.93
5        34.94  65.06

Marriage distribution by cluster (%)
marriage      0     1      2      3
cluster                            
0          2.80  1.14  84.91  11.14
1         74.39  7.57  15.66   2.38
2         80.72  6.76  10.94   1.58
3         73.28  5.51  17.20   4.02
4          9.58  0.79 

In [40]:
## 5. Second Cluster Result
## After the initial clustering analyses, 
## we observed that the clustering solution was highly sensitive to the encoding strategy for categorical variables.
## In particular, full one-hot encoding led to a gender-dominated solution, 
## while reduced-redundancy one-hot encoding produced clusters that were still strongly stratified by education level.
## Because education and income are ordinal rather than purely nominal variables, 
## a third specification was introduced in which ordinal variables were retained in their ranked form and standardized, 
## while only nominal variables were one-hot encoded.
## This approach was adopted to better align the data preprocessing strategy 
## with the measurement properties of the variables and to obtain a more balanced and interpretable clustering structure.

In [42]:
## 6. Third Cluster With Ordinal-aware
df_v3 = df_cluster[['age', 'marriage', 'income', 'education', 'gender']].copy()

df_v3_encoded = pd.get_dummies(
    df_v3,
    columns=['marriage'],
    drop_first=True
)

# ordinal/continuous
scale_cols_v3 = ['age', 'income', 'education']

scaler_v3 = StandardScaler()
df_v3_encoded[scale_cols_v3] = scaler_v3.fit_transform(df_v3_encoded[scale_cols_v3])

# bool to int
bool_cols_v3 = df_v3_encoded.select_dtypes(include='bool').columns
df_v3_encoded[bool_cols_v3] = df_v3_encoded[bool_cols_v3].astype(int)

# gender 0/1
df_v3_encoded['gender'] = pd.to_numeric(df_v3_encoded['gender'], errors='coerce')

X_v3 = df_v3_encoded.copy()

print(X_v3.shape)
print(X_v3.dtypes)
print(X_v3.head())

# K Scan
X_v3_sample = X_v3.sample(n=5000, random_state=42)

k_values = range(2, 9)
silhouette_scores_v3 = []
inertias_v3 = []
cluster_sizes_v3 = {}

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_full = kmeans.fit_predict(X_v3)
    labels_sample = kmeans.predict(X_v3_sample)

    sil = silhouette_score(X_v3_sample, labels_sample)
    inertia = kmeans.inertia_

    silhouette_scores_v3.append(sil)
    inertias_v3.append(inertia)
    cluster_sizes_v3[k] = pd.Series(labels_full).value_counts().sort_index()

    print(f"\nK = {k}")
    print(f"Silhouette Score = {sil:.4f}")
    print(f"Inertia = {inertia:.2f}")
    print("Cluster sizes:")
    print(cluster_sizes_v3[k])



(30000, 7)
age           float64
income        float64
education     float64
gender          int64
marriage_1      int64
marriage_2      int64
marriage_3      int64
dtype: object
        age    income  education  gender  marriage_1  marriage_2  marriage_3
0  0.011510  2.340148   0.926088       1           0           0           0
1  1.261652  1.302051   1.731756       1           0           1           0
2 -1.863703 -0.774144  -0.685247       0           0           0           0
3 -1.988718 -0.774144  -1.490915       1           0           0           0
4 -1.988718 -0.774144  -1.490915       1           0           0           0

K = 2
Silhouette Score = 0.3078
Inertia = 73663.27
Cluster sizes:
0    10458
1    19542
Name: count, dtype: int64

K = 3
Silhouette Score = 0.2597
Inertia = 58551.15
Cluster sizes:
0     8176
1    12172
2     9652
Name: count, dtype: int64

K = 4
Silhouette Score = 0.2805
Inertia = 48479.95
Cluster sizes:
0    12202
1     5172
2     4588
3     8038
Name: c

In [44]:
## 6. Third Cluter Result
## To improve the substantive interpretability of the clustering solution,
## a third specification was implemented using an ordinal-aware encoding strategy.
## Specifically, age, education, and income were treated as continuous/ordinal variables 
## and standardized, gender was retained as a binary variable, and only marriage was one-hot encoded.
## This specification better reflected the measurement properties of the variables 
## and reduced the dominance of individual categorical dimensions observed in earlier models.

## Selecttion of K
## Across the ordinal-aware clustering solutions, 
## the 4-cluster model achieved the highest silhouette score among the multi-cluster candidates (0.2805), 
## while also maintaining adequate cluster sizes and a manageable level of interpretive complexity.
## Although the 2-cluster solution had the highest overall silhouette score, it was considered too coarse for meaningful subgroup profiling. 
## Therefore, the 4-cluster solution was selected as the final model.

In [46]:
## 7. Final Model
# Final model: Version 3 + K=4
k_final_v3 = 4
kmeans_final_v3 = KMeans(n_clusters=k_final_v3, random_state=42, n_init=10)

df_v3_profile = df_cluster[['age', 'marriage', 'income', 'education', 'gender']].copy()
df_v3_profile['cluster'] = kmeans_final_v3.fit_predict(X_v3)

# counts
cluster_counts_v3 = df_v3_profile['cluster'].value_counts().sort_index()
cluster_perc_v3 = df_v3_profile['cluster'].value_counts(normalize=True).sort_index() * 100

print("Cluster counts:")
print(cluster_counts_v3)

print("\nCluster percentages:")
print(cluster_perc_v3.round(2))

# age summary
age_summary_v3 = df_v3_profile.groupby('cluster')['age'].agg(['mean', 'median', 'std', 'min', 'max', 'count'])
print("\nAge summary:")
print(age_summary_v3.round(2))

# distribution tables
def cluster_distribution_table(df, cluster_col, var):
    table = pd.crosstab(df[cluster_col], df[var], normalize='index') * 100
    return table.round(2)

print("\nGender distribution by cluster (%)")
print(cluster_distribution_table(df_v3_profile, 'cluster', 'gender'))

print("\nMarriage distribution by cluster (%)")
print(cluster_distribution_table(df_v3_profile, 'cluster', 'marriage'))

print("\nIncome distribution by cluster (%)")
print(cluster_distribution_table(df_v3_profile, 'cluster', 'income'))

print("\nEducation distribution by cluster (%)")
print(cluster_distribution_table(df_v3_profile, 'cluster', 'education'))

Cluster counts:
cluster
0    12202
1     5172
2     4588
3     8038
Name: count, dtype: int64

Cluster percentages:
cluster
0    40.67
1    17.24
2    15.29
3    26.79
Name: proportion, dtype: float64

Age summary:
          mean  median   std   min   max  count
cluster                                        
0        22.22    22.0  3.60  10.0  36.0  12202
1        37.51    37.0  5.49  28.0  48.0   5172
2        31.13    30.0  7.17  12.0  48.0   4588
3        21.06    21.0  4.91  10.0  36.0   8038

Gender distribution by cluster (%)
gender       0      1
cluster              
0        22.18  77.82
1        33.31  66.69
2        43.55  56.45
3        33.03  66.97

Marriage distribution by cluster (%)
marriage      0     1      2     3
cluster                           
0         74.36  6.34  17.69  1.61
1          4.04  0.83  86.58  8.55
2         24.04  4.45  65.19  6.32
3         62.39  5.56  27.63  4.42

Income distribution by cluster (%)
income       0      1      2      3    4     

In [48]:
## 7. Final Model Result
## Cluster 0(41%) : Young single lower-income college/university men
##                  22 years old, Male(78%), Single(74%), 0-50k/yr(64%), College&University(95%)
## Cluster 1(17%) : Older married middle-income adults
##                  37 years old, Married(87%), 50-100k/yr(57.21%), Male(67%)
## Cluster 2(15%) : Highly educated higher-income married adults
##                  31 years old, Married(65%), 100-200k/yr(63%), University&Master&Doctorate(73%), Male(56%), Female(43%)
## Cluster 3(27%) : Very young lower-education low-income singles
##                  21 years old, Single(63%), 0-50k/yr(74%), Middle/High School(93%), Male(67%)

In [54]:
## 8. Final Model Anxiety Score Test
df_v3_profile = df_cluster[['age', 'marriage', 'income', 'education', 'gender']].copy()
df_v3_profile['score'] = df['score'].values
df_v3_profile['cluster'] = kmeans_final_v3.fit_predict(X_v3)

score_summary = df_v3_profile.groupby('cluster')['score'].agg(['mean', 'median', 'std', 'min', 'max', 'count'])
print(score_summary.round(2))

from scipy.stats import kruskal

groups = [group['score'].dropna().values for _, group in df_v3_profile.groupby('cluster')]
stat, p_value = kruskal(*groups)

print(f"Kruskal-Wallis H = {stat:.4f}")
print(f"p-value = {p_value:.6f}")

def gad7_severity(score):
    if score <= 4:
        return 'minimal'
    elif score <= 9:
        return 'mild'
    elif score <= 14:
        return 'moderate'
    else:
        return 'severe'

df_v3_profile['severity'] = df_v3_profile['score'].apply(gad7_severity)

severity_table = pd.crosstab(
    df_v3_profile['cluster'],
    df_v3_profile['severity'],
    normalize='index'
) * 100

severity_table = severity_table[['minimal', 'mild', 'moderate', 'severe']]

print(severity_table.round(2))

         mean  median   std  min  max  count
cluster                                     
0        8.62     8.0  5.62    0   21  12202
1        7.24     6.0  5.63    0   21   5172
2        6.85     6.0  5.55    0   21   4588
3        9.89     9.0  6.02    0   21   8038
Kruskal-Wallis H = 1110.5702
p-value = 0.000000
severity  minimal   mild  moderate  severe
cluster                                   
0           25.50  35.85     20.73   17.92
1           35.81  34.44     16.05   13.71
2           38.75  34.37     14.82   12.05
3           20.49  30.98     22.75   25.78


In [ ]:
## 8. Final Model Result
## GAD-7 scores differed significantly across the four demographic clusters (Kruskal-Wallis H = 1110.57, p < 0.001). 
## Cluster 3 showed the highest anxiety burden,
## with the highest mean score (9.89) and median score (9), followed by Cluster 0 (mean = 8.62, median = 8).
## In contrast, Clusters 1 and 2 showed lower anxiety levels, with Cluster 2 having the lowest mean score (6.85) and median score (6).
## Severity distributions further supported this pattern. 
## Cluster 3 had the highest proportions of moderate (22.75%) and severe anxiety (25.78%), 
## whereas Cluster 2 had the highest proportion of minimal anxiety (38.75%) and the lowest proportion of severe anxiety (12.05%). 
## Cluster 0 also showed elevated anxiety burden, while Clusters 1 and 2 represented comparatively lower-risk groups.

## Interpretation
## These findings indicate that the final demographic clustering solution is meaningfully associated with anxiety severity.
## The subgroup with the highest anxiety burden was characterized by 
## younger age, lower education, lower income, and a high proportion of single respondents, 
## suggesting that socioeconomic disadvantage may compound anxiety risk within younger populations.
## By contrast, the lowest anxiety burden was observed in the subgroup with higher 
## education, higher income, and more stable marital status,
## pointing to a potentially protective role of greater socioeconomic stability.
## Notably, although both Cluster 0 and Cluster 3 represented relatively young groups, 
## Cluster 3 consistently showed worse anxiety outcomes, 
## indicating that younger age alone does not fully explain the observed differences 
## and that educational and economic disadvantage may further intensify anxiety burden.